# Feature Selection
Reduce the full 72-feature set down to 25 features confirmed through Mann-Whitney analysis.
Drops redundant, correlated, and statistically weak features.
Produces a clean dataset ready for train/test split and modelling.
Unknown recordings are carried through separately, they are never used in analysis or training.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT  = Path(r"D:\sop")
FEATURES_DIR  = PROJECT_ROOT / "data" / "features"

print("Ready.")

Ready.


In [2]:
# loading the full 72-feature dataset

df_all = pd.read_csv(FEATURES_DIR / "features.csv")

df_labelled = df_all[df_all["class"].isin(["absent", "present"])].copy().reset_index(drop=True)
df_unknown  = df_all[df_all["class"] == "unknown"].copy().reset_index(drop=True)

print(f"Total files     : {len(df_all)}")
print(f"Labelled        : {len(df_labelled)}  (absent={len(df_labelled[df_labelled['class']=='absent'])}, present={len(df_labelled[df_labelled['class']=='present'])})")
print(f"Unknown (held)  : {len(df_unknown)}")
print(f"Full feature set: {len([c for c in df_all.columns if c not in ['file','class','label']])} features")

Total files     : 7173
Labelled        : 4782  (absent=2391, present=2391)
Unknown (held)  : 2391
Full feature set: 72 features


In [ ]:
# confirmed feature list : 25 features selected through Mann-Whitney analysis
# all features have effect_r >= 0.06 and p < 0.001
#
# added selection:
#   kraskov entropy (all 4 modes) : top 2 features overall, effect_r up to 0.354
#   sample entropy (u3, u4)       : effect_r 0.130 and 0.200 respectively
#
# dropped selection:
#   u1_spectral_bandwidth : effect_r=0.053, too weak
#   omega3, omega4        : effect_r=0.009 and 0.035, not meaningfully significant
#   v_jump_std            : effect_r=0.094, weakest jump feature, redundant with v_jump_mean
#
# still excluded:
#   mean        : near-zero for all recordings, no separation
#   rms         : redundant with variance (r > 0.91 across all modes)
#   zcr         : redundant with omega (r > 0.999)
#   skewness    : effect_r < 0.04, negligible
#   dom_freq    : redundant with omega (r > 0.998)
#   centroid    : redundant with omega (r > 0.99)
#   spectral_entropy    : effect_r < 0.06 across all modes
#   energy_ratio        : effect_r < 0.03, no separation
#   perm_entropy        : effect_r < 0.09, weak and noisy
#   u1/u2 sample_entropy : effect_r < 0.05, no meaningful separation
#   v_jump_max  : effect_r=0.026, negligible
#   omega1 spread/weighted_mean : redundant with individual omegas

SELECTED_FEATURES = [
    # variance : amplitude swing per mode (effect_r ~0.23 across all modes)
    "u1_var", "u2_var", "u3_var", "u4_var",

    # kurtosis : waveform spikiness, murmurs smooth out sharp peaks (effect_r up to 0.27)
    "u1_kurt", "u2_kurt", "u3_kurt", "u4_kurt",

    # mode energy : total oscillatory power per mode (effect_r ~0.22)
    "u1_energy", "u2_energy", "u3_energy", "u4_energy",

    # kraskov entropy : information content per mode, strongest new features (effect_r up to 0.354)
    "u1_kraskov_entropy", "u2_kraskov_entropy", "u3_kraskov_entropy", "u4_kraskov_entropy",

    # sample entropy : signal regularity, murmurs are less regular (effect_r 0.130 - 0.200)
    "u3_sample_entropy", "u4_sample_entropy",

    # omega : center frequency of the two most discriminative modes
    "omega1", "omega2",

    # spectral bandwidth : frequency spread of highest mode (effect_r=0.096)
    "u4_spectral_bandwidth",

    # jump features : abrupt discontinuities from valve events and turbulence
    "v_total_variation", # total accumulated jump change (effect_r=0.175)
    "v_energy",          # power in jump component (effect_r=0.122)
    "v_jump_mean",       # average jump size (effect_r=0.121)
    "v_num_jumps",       # count of abrupt events, present has ~2x absent (effect_r=0.116)
]

print(f"Selected features: {len(SELECTED_FEATURES)}")
for f in SELECTED_FEATURES:
    print(f"  {f}")

Selected features: 25
  u1_var
  u2_var
  u3_var
  u4_var
  u1_kurt
  u2_kurt
  u3_kurt
  u4_kurt
  u1_energy
  u2_energy
  u3_energy
  u4_energy
  u1_kraskov_entropy
  u2_kraskov_entropy
  u3_kraskov_entropy
  u4_kraskov_entropy
  u3_sample_entropy
  u4_sample_entropy
  omega1
  omega2
  u4_spectral_bandwidth
  v_total_variation
  v_energy
  v_jump_mean
  v_num_jumps


In [4]:
# verifying all selected features exist in the dataset

missing = [f for f in SELECTED_FEATURES if f not in df_all.columns]
if missing:
    print(f"ERROR : features not found in dataset: {missing}")
else:
    print(f"All {len(SELECTED_FEATURES)} selected features verified present in dataset.")

All 25 selected features verified present in dataset.


In [5]:
# build reduced datasets - labelled and unknown
# keep file and class columns alongside features for traceability

keep_cols = ["file", "class"] + SELECTED_FEATURES

df_labelled_reduced = df_labelled[keep_cols].copy()
df_unknown_reduced  = df_unknown[keep_cols].copy()

print(f"Labelled reduced shape  : {df_labelled_reduced.shape}")
print(f"Unknown reduced shape   : {df_unknown_reduced.shape}")
print(f"\nClass distribution in labelled:")
print(df_labelled_reduced["class"].value_counts().to_string())

Labelled reduced shape  : (4782, 27)
Unknown reduced shape   : (2391, 27)

Class distribution in labelled:
class
absent     2391
present    2391


In [6]:
# quick sanity check - confirm no nulls introduced during selection

null_counts = df_labelled_reduced[SELECTED_FEATURES].isnull().sum()
if null_counts.sum() == 0:
    print("No null values in reduced labelled dataset.")
else:
    print("Null values found:")
    print(null_counts[null_counts > 0])

null_counts_u = df_unknown_reduced[SELECTED_FEATURES].isnull().sum()
if null_counts_u.sum() == 0:
    print("No null values in reduced unknown dataset.")
else:
    print("Null values found in unknown:")
    print(null_counts_u[null_counts_u > 0])

No null values in reduced labelled dataset.
No null values in reduced unknown dataset.


In [7]:
# save reduced datasets
# labelled  : used for train/test split and model training
# unknown   : held out, used only for final prediction

out_labelled = FEATURES_DIR / "features_selected.csv"
out_unknown  = FEATURES_DIR / "features_selected_unknown.csv"

df_labelled_reduced.to_csv(out_labelled, index=False)
df_unknown_reduced.to_csv(out_unknown,  index=False)

print(f"Saved: {out_labelled}")
print(f"Saved: {out_unknown}")
print(f"\nFull feature set : 72 features")
print(f"Selected         : {len(SELECTED_FEATURES)} features")
print(f"Dropped          : {72 - len(SELECTED_FEATURES)} features")

Saved: D:\sop\data\features\features_selected.csv
Saved: D:\sop\data\features\features_selected_unknown.csv

Full feature set : 72 features
Selected         : 25 features
Dropped          : 47 features
